# DMN Colab - Live Train / Predict

Notebook Colab pour entrainer un artefact DMN LSTM et predire des positions futures via `python -m dmn.cli.live`, en persistant les fichiers sur Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/figaroldamien/DMN.git"
REPO_DIR = Path("/content/DMN")
DRIVE_DIR = Path("/content/drive/MyDrive/DMN")
ARTIFACT_DIR = DRIVE_DIR / "artifacts" / "dmn"
OUTPUT_DIR = DRIVE_DIR / "output"
LOGS_DIR = DRIVE_DIR / "logs"

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "notebooks"))
from notebook_logging import run_logged

print(f"Working directory: {REPO_DIR}")
print(f"Drive root: {DRIVE_DIR}")
print(f"Logs directory: {LOGS_DIR}")


In [ ]:
COMMAND = "predict"  # "train" or "predict"
MARKET = "cac40"
TICKER = None
START = "2000-01-01"
SECTOR = None
SUB_SECTOR = None

CUTOFF_MODE = "year_end_prev"
CUTOFF_DATE = None
REFERENCE_DATE = None
SEQ_LEN = 63
HIDDEN = 32
DROPOUT = 0.1
USE_TICKER_EMBEDDING = True
LR = 1e-3
EPOCHS = 20
BATCH_SIZE = 256
TURNOVER_LAMBDA = 0.0
SEED = 0
MIN_TRAIN_SAMPLES = 2000

ARTIFACT_PATH = str(ARTIFACT_DIR / "cac40_20251231.pt")
FROM_DATE = "2026-01-01"
TO_DATE = None
OUTPUT_PATH = str(OUTPUT_DIR / "cac40_predictions.csv")
LOG_PATH = str(LOGS_DIR / f"live_{COMMAND}.log")
PRINT_CONFIG = True


In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = f"{REPO_DIR / 'src'}:{env.get('PYTHONPATH', '')}"

cmd = [sys.executable, "-m", "dmn.cli.live", COMMAND]
if TICKER:
    cmd.extend(["--ticker", TICKER])
else:
    cmd.extend(["--market", MARKET])
if START:
    cmd.extend(["--start", START])
if SECTOR:
    cmd.extend(["--sector", SECTOR])
if SUB_SECTOR:
    cmd.extend(["--sub-sector", SUB_SECTOR])

if COMMAND == "train":
    cmd.extend(["--cutoff-mode", CUTOFF_MODE, "--artifact-dir", str(ARTIFACT_DIR)])
    if CUTOFF_DATE:
        cmd.extend(["--cutoff-date", CUTOFF_DATE])
    if REFERENCE_DATE:
        cmd.extend(["--reference-date", REFERENCE_DATE])
    cmd.extend([
        "--seq-len", str(SEQ_LEN),
        "--hidden", str(HIDDEN),
        "--dropout", str(DROPOUT),
        "--lr", str(LR),
        "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--turnover-lambda", str(TURNOVER_LAMBDA),
        "--seed", str(SEED),
        "--min-train-samples", str(MIN_TRAIN_SAMPLES),
    ])
    cmd.append("--use-ticker-embedding" if USE_TICKER_EMBEDDING else "--no-use-ticker-embedding")
elif COMMAND == "predict":
    cmd.extend(["--artifact-path", ARTIFACT_PATH, "--output", OUTPUT_PATH])
    if FROM_DATE:
        cmd.extend(["--from-date", FROM_DATE])
    if TO_DATE:
        cmd.extend(["--to-date", TO_DATE])
else:
    raise ValueError("COMMAND must be 'train' or 'predict'")

if not PRINT_CONFIG:
    cmd.append("--no-print-config")

run_logged(cmd, env=env, log_path=LOG_PATH)
